## Finetuning T5 for MetaData Generation

### Logging In to Hub

In [1]:
from huggingface_hub import notebook_login, whoami

notebook_login()

In [2]:
whoami()

{'type': 'user',
 'id': '66a797b589b3e71262932d0d',
 'name': 'SurAyush',
 'fullname': 'Ayush Sur',
 'email': 'ayushsur26@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'periodEnd': None,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/RZJZW_w0wdVoOmQY250lR.png',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Google Colab',
   'role': 'write',
   'createdAt': '2025-03-02T10:56:32.713Z'}}}

### Loading and Preprocessing Dataset

In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset('SurAyush/ProductTitle-To-Category')

raw_datasets

README.md:   0%|          | 0.00/436 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/58.1M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/12.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/443499 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/95035 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 443499
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 95035
    })
})

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

checkpoint = 'google-t5/t5-small'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
# testing tokenizer

out = tokenizer("Hello there! I am Ayush Sur")
print(out)
print(tokenizer.convert_ids_to_tokens(out.input_ids))
print(tokenizer.decode(out['input_ids']))

{'input_ids': [8774, 132, 55, 27, 183, 71, 63, 8489, 3705, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['▁Hello', '▁there', '!', '▁I', '▁am', '▁A', 'y', 'ush', '▁Sur', '</s>']
Hello there! I am Ayush Sur</s>


In [6]:
# model size

print(model.num_parameters()/1_000_000, "M")

60.506624 M


In [7]:
def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        max_length = 384
    )

    # same tokenizer for labels
    labels = tokenizer(
        examples["target"],
        truncation=True,
        max_length = 384
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [8]:
# tokenizing dataset

tokenized_dataset = raw_datasets.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/443499 [00:00<?, ? examples/s]

Map:   0%|          | 0/95035 [00:00<?, ? examples/s]

In [9]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 443499
    })
    validation: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 95035
    })
})

In [10]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [11]:
tokenized_dataset = tokenized_dataset.remove_columns(
    raw_datasets["train"].column_names
)

In [12]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 443499
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 95035
    })
})

### Setting Up Training

In [1]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import get_scheduler

In [13]:
from accelerate import Accelerator

In [14]:
tokenized_dataset.set_format("torch")       # setting up dataset to torch format

#### Overfitting a single batch to check capabilites of T5 small for the task

In [17]:
import numpy as np

optimizer = AdamW(model.parameters(), lr=1e-3)

train_dataloader = DataLoader(
    tokenized_dataset["train"].select(range(16)),       # only a subset of 16
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_dataset["train"].select(range(16)),
    collate_fn=data_collator,
    batch_size=8
)

In [18]:
for epoch in range(20):
  # Training
  model.train()
  loss_epoch = []
  for step, batch in enumerate(train_dataloader):
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    loss_epoch.append(loss.item())

    optimizer.step()
    optimizer.zero_grad()

  print("Epoch: ", epoch, "Loss: ", np.mean(loss_epoch))


  # Eval
  model.eval()
  for step, batch in enumerate(eval_dataloader):
    with torch.no_grad():
      generated_tokens = model.generate(
          batch['input_ids'],
          attention_mask=batch['attention_mask'],
          max_length=384
      )
      labels = batch["labels"]



      generated_tokens = generated_tokens.cpu().numpy()
      labels = labels.cpu().numpy()

      # Replace -100 in the labels as we can't decode them
      labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

      if isinstance(generated_tokens, tuple):
          generated_tokens = generated_tokens[0]

      decoded_preds = tokenizer.batch_decode(
          generated_tokens, skip_special_tokens=True
      )
      decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

      print("Predictions: ", decoded_preds)
      print("Actual: ", decoded_labels)

/usr/local/lib/python3.11/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch:  0 Loss:  3.498453974723816
Predictions:  ['Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume Manufacturer, Enclume', 'Schutt Vengeance DCT Hybrid Youth Football H Store: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Schutt Manufacturer: Sch

Not very staisfory results in 20 epcohs but reduction in loss and appearence of output pattern in okay to proceed with

#### Training Loop

In [15]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [16]:
import numpy as np

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=32,
)
eval_dataloader = DataLoader(
    tokenized_dataset["validation"],
    collate_fn=data_collator,
    batch_size=32
)

In [21]:
optimizer = AdamW(model.parameters(), lr=1e-4)

In [22]:
accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [23]:
num_train_epochs = 2
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=1200,
    num_training_steps=num_training_steps,
)

In [24]:
from huggingface_hub import get_full_repo_name

model_name = "title2metadata"
repo_name = get_full_repo_name(model_name)
repo_name

'SurAyush/title2metadata'

In [25]:
from huggingface_hub import Repository

output_dir = "title2metadata"
repo = Repository(output_dir, clone_from=repo_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
Cloning https://huggingface.co/SurAyush/title2metadata into local empty directory.


In [26]:
# custom training loop
import random

progress_bar = tqdm(range(num_training_steps))
loss_log_steps =2000

for epoch in range(num_train_epochs):

  # Training
  model.train()

  loss_epoch = []
  for step, batch in enumerate(train_dataloader):
    outputs = model(**batch)
    loss = outputs.loss
    accelerator.backward(loss)
    loss_epoch.append(loss.item())

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)

    if step % loss_log_steps == 0:
      print(f"Epoch: {epoch}  Steps: {step} Loss Avg: {np.mean(loss_epoch[-loss_log_steps:])}")


  print("Epoch: ", epoch, "Loss: ", np.mean(loss_epoch))

  with open('train_loss_hist.txt','a') as file:
    for l in loss_epoch:
      file.write(f"{l},")


  # Evalulation
  loss_epoch = []
  model.eval()
  random_step = random.randint(0,2000)    # ~1400 steps in val

  for step, batch in enumerate(eval_dataloader):
    with torch.no_grad():
      outputs = model(**batch)
      loss = outputs.loss
      loss_epoch.append(loss.item())

      # logging some generations (of one batch)
      if step == random_step:
        generated_tokens = model.generate(
            batch['input_ids'],
            attention_mask=batch['attention_mask'],
            max_length=384
        )
        labels = batch["labels"]

        generated_tokens = generated_tokens.cpu().numpy()
        labels = labels.cpu().numpy()

        # Replace -100 in the labels as we can't decode them
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        if isinstance(generated_tokens, tuple):
            generated_tokens = generated_tokens[0]

        decoded_preds = tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True
        )
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        print("Predictions: ", decoded_preds)
        print("Actual: ", decoded_labels)


  print("Epoch: ", epoch, "Eval Loss: ", np.mean(loss_epoch))
  with open('val_loss_hist.txt','a') as file:
    for l in loss_epoch:
      file.write(f"{l},")

  0%|          | 0/27720 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch: 0  Steps: 0 Loss Avg: 3.957047462463379
Epoch: 0  Steps: 2000 Loss Avg: 0.8913389284014702
Epoch: 0  Steps: 4000 Loss Avg: 0.24928640903532506
Epoch: 0  Steps: 6000 Loss Avg: 0.16825873006880282
Epoch: 0  Steps: 8000 Loss Avg: 0.12736796572059392
Epoch: 0  Steps: 10000 Loss Avg: 0.10434068719483912
Epoch: 0  Steps: 12000 Loss Avg: 0.0879136918708682
Epoch:  0 Loss:  0.24552551291393812
Predictions:  ['"details_Brand": "Royal Canin", "L0_category": "Pet Supplies", "L1_category": "Dogs", "L2_category": "Food", "L3_category": "Dry", "L4_category": "na"', '"details_Brand": "McCall\'s Patterns", "L0_category": "Arts, Crafts & Sewing", "L1_category": "Sewing", "L2_category": "Sewing Patterns & Templates", "L3_category": "na", "L4_category": "na"', '"details_Brand": "Parker", "L0_category": "Office Products", "L1_category": "Office & School Supplies", "L2_category": "Writing & Correction Supplies", "L3_category": "Pens & Refills", "L4_category": "Fountain Pens"', '"details_Brand": "Rac

KeyboardInterrupt: 

Predictions are good enough after single epoch so interrupting training...

#### Saving the model to Hub

In [27]:
accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
if accelerator.is_main_process:
  tokenizer.save_pretrained(output_dir)
  repo.push_to_hub(
      commit_message=f"Training complete", blocking=False
  )

In [28]:
# saving model file locally to
torch.save(unwrapped_model.state_dict(), 'model.pth')

### Testing the Model Pipeline

In [29]:
from transformers import pipeline

pipe = pipeline("text2text-generation", model = unwrapped_model, tokenizer=tokenizer)

Device set to use cuda:0


In [31]:
input_text = raw_datasets['train'][9]['input']

In [32]:
input_text

'Classify product:\nTitle: MSI LGA1155/Intel H61 B3/DDR3/A&GbE/MATX Motherboard H61M-P23 (B3)\nStore: msi\nManufacturer: MSI Computer Corp.'

In [33]:
result = pipe(input_text, max_length=384, clean_up_tokenization_spaces=True)

print(result[0]['generated_text'])

"details_Brand": "msi", "L0_category": "Electronics", "L1_category": "Computers & Accessories", "L2_category": "Computer Components", "L3_category": "Internal Components", "L4_category": "Motherboards"


In [35]:
raw_datasets['train'][9]['target']

'{"details_Brand": "MSI", "L0_category": "Electronics", "L1_category": "Computers & Accessories", "L2_category": "Computer Components", "L3_category": "Internal Components", "L4_category": "Motherboards"}'

In [37]:
# Testing model-uploaded to hub
pipe2 = pipeline('text2text-generation', 'SurAyush/title2metadata')
result = pipe2(input_text, max_length=384, clean_up_tokenization_spaces=True)

print(result[0]['generated_text'])

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Device set to use cuda:0


"details_Brand": "MSI", "L0_category": "Electronics", "L1_category": "Computers & Accessories", "L2_category": "Computer Components", "L3_category": "Internal Components", "L4_category": "Motherboards"


In [36]:
samples = raw_datasets['validation'].shuffle().select(range(10))

for sample in samples:
  print("Input: ", sample['input'])
  print("Actual: ", sample['target'])
  result = pipe(sample['input'], max_length=384, clean_up_tokenization_spaces=True)
  print(result[0]['generated_text'])

Input:  Classify product:
Title: Mtd 731-09728 Lawn Mower Ignition Key Genuine Original Equipment Manufacturer (OEM) Part
Store: MTD Genuine Parts
Manufacturer: Mtd
Actual:  {"details_Brand": "MTD Genuine Parts", "L0_category": "Patio, Lawn & Garden", "L1_category": "Outdoor Power Tools", "L2_category": "Replacement Parts & Accessories", "L3_category": "Lawn Mower Parts & Accessories", "L4_category": "Lawn Mower Replacement Parts"}
"details_Brand": "MTD Genuine Parts", "L0_category": "Patio, Lawn & Garden", "L1_category": "Outdoor Power Tools", "L2_category": "Replacement Parts & Accessories", "L3_category": "Lawn Mower Parts & Accessories", "L4_category": "Lawn Mower Replacement Parts"
Input:  Classify product:
Title: NHL 12 Can Soft Sided Cooler
Store: Coleman
Manufacturer: The Licensed Products Company
Actual:  {"details_Brand": "Coleman", "L0_category": "Sports & Outdoors", "L1_category": "Fan Shop", "L2_category": "Patio, Lawn & Garden", "L3_category": "Coolers", "L4_category": "n

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


"details_Brand": "Crush", "L0_category": "Office Products", "L1_category": "Office & School Supplies", "L2_category": "Store Signs & Displays", "L3_category": "Store Signs", "L4_category": "na"
Input:  Classify product:
Title: Oregon OEM 15-126 Belt Replaces Grasshoppe[265] Grasshopper - 382093 Rotary - 14363
Store: Oregon
Manufacturer: Oregon
Actual:  {"details_Brand": "Oregon", "L0_category": "Patio, Lawn & Garden", "L1_category": "Outdoor Power Tools", "L2_category": "Replacement Parts & Accessories", "L3_category": "Lawn Mower Parts & Accessories", "L4_category": "Lawn Mower Replacement Parts"}
"details_Brand": "Oregon", "L0_category": "Industrial & Scientific", "L1_category": "Power Transmission Products", "L2_category": "Belts", "L3_category": "Bittles", "L4_category": "na"


#### We did it!!!